# Rate-limit benchmark audit

This notebook compares serialized `survey_sim` results with the ZTF literature benchmark from Andreoni et al. (2020), arXiv:2008.00008v1. The paper reports 95% limits of 1,775 Gpc^-3 yr^-1 for a fixed GW170817-like viewing angle and 4,029 Gpc^-3 yr^-1 for isotropic viewing angles. The comparison is diagnostic, not a claim that the survey baselines or model grids are identical.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV = Path('results/rate_limit_benchmark.csv')
data = pd.read_csv(CSV)
data['model'] = data['file'].str.extract(r'results/marathon/([^/]+)/')[0].fillna(data['file'].str.extract(r'results/([^/]+)/')[0])
data.shape

In [ ]:
published_fixed = 1775.0
published_isotropic = 4029.0
columns = ['model', 'rate_95_zero_events', 'duration_scaled_95', 'effective_vt_gpc3_yr', 'survey_duration_years', 'overall_efficiency']
data.groupby('model')[columns[1:]].agg(['count', 'median', 'min', 'max']).round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for model, frame in data.groupby('model'):
    ax.hist(frame['duration_scaled_95'], bins=30, alpha=0.45, label=model)
ax.axvline(published_fixed, color='black', linestyle='--', label='published fixed angle')
ax.axvline(published_isotropic, color='black', linestyle=':', label='published isotropic')
ax.set_xlabel('95% limit scaled to 23 months [Gpc^-3 yr^-1]')
ax.set_ylabel('number of result files')
ax.set_xscale('log')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for model, frame in data.groupby('model'):
    axes[0].scatter(frame['effective_vt_gpc3_yr'], frame['overall_efficiency'], s=8, alpha=0.35, label=model)
    axes[1].scatter(frame['survey_duration_years'], frame['rate_95_zero_events'], s=8, alpha=0.35, label=model)
axes[0].set_xlabel('effective VT [Gpc^3 yr]')
axes[0].set_ylabel('overall MC efficiency')
axes[0].set_xscale('log')
axes[1].set_xlabel('survey duration [yr]')
axes[1].set_ylabel('raw 95% zero-event limit [Gpc^-3 yr^-1]')
axes[1].set_yscale('log')
axes[1].legend()
for axis in axes:
    axis.grid(alpha=0.3)
fig.tight_layout()

In [ ]:
# Find runs whose VT differs strongly from the current fixed-model reproduction.
data.sort_values('effective_vt_gpc3_yr').loc[:, ['model', 'file', 'effective_vt_gpc3_yr', 'rate_95_zero_events', 'z_max', 'survey_duration_years']].head(20)

## Interpretation checklist

1. Compare fixed-angle results with the 1,775 benchmark before interpreting randomized or isotropic results.
2. Inspect `effective_vt_gpc3_yr` directly; the 95% zero-event numerator is fixed at 2.996, so a high limit means low VT.
3. Check survey duration and model provenance. A rate limit cannot be compared across baselines without accounting for elapsed survey time and cadence.
4. Treat malformed JSON or mixed result directories as an analysis failure, not as a physical outlier.